In [ ]:
# ---- 0. 环境设置 -----------------------------------------------------------
Sys.setenv(SP_EVOLUTION_STATUS = "2")
set.seed(1)

suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(stringr)
  library(AnnotationDbi)
  library(org.Hs.eg.db)
  library(fgsea)
  library(msigdbr)
})

# ---- 1. 参数设置 -----------------------------------------------------------
date_str <- "2026-08-02_1"

base_dir <- "/project/work/2026-07-29_trajectory_class/"
data_dir <- file.path(base_dir, "Data")
dev_dir <- "/project/work/2026-07-06_development_DEG/Data"
adult_dir <- "/project/work/2026-06-10_adult_DEG/Data/02_regional_degs"
trajectory_file <- file.path(
  data_dir,
  "2026-07-29_1_five_stage_regional_DEG_trajectory_classified.csv"
)

min_size <- 15L
max_size <- 500L
padj_cutoff <- 0.05
min_abs_nes_drop <- 0.25

stage_order <- c("Early", "Mid", "Late", "Young", "Old")
celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
available_stages <- list(
  AC = stage_order,
  Cone = stage_order,
  HC = stage_order,
  Rod = c("Mid", "Late", "Young", "Old"),
  MG = c("Mid", "Late", "Young", "Old"),
  BC = c("Late", "Young", "Old")
)
target_subclasses <- c(
  "Developmental maintained",
  "Developmental aging-lost",
  "Adult acquired and maintained"
)
direction_labels <- c(
  Central_macular = "Macular direction",
  Peripheral = "Peripheral direction"
)

required_packages <- c(
  "dplyr", "tidyr", "stringr", "AnnotationDbi", "org.Hs.eg.db",
  "fgsea", "msigdbr"
)
missing_packages <- required_packages[
  !vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)
]
if (length(missing_packages) > 0) {
  stop(
    "缺少 R 包：", paste(missing_packages, collapse = ", "),
    "。请在服务器 sc 环境安装后再运行。",
    call. = FALSE
  )
}

dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# ---- 2. 读取完整 DESeq2 结果并检查 contrast --------------------------------
dev_files <- list.files(
  dev_dir,
  pattern = "^2026-07-06_2_.*_(Early|Mid|Late)_macula_vs_peripheral\\.csv$",
  full.names = TRUE
)
adult_files <- list.files(
  adult_dir,
  pattern = "^2026-07-28_adult_.*_(Young|Old)_Central_vs_peripheral\\.csv$",
  full.names = TRUE
)

if (length(dev_files) == 0 || length(adult_files) == 0) {
  stop("未找到完整胚胎或成年/老年 DESeq2 结果文件。", call. = FALSE)
}
if (!file.exists(trajectory_file)) {
  stop("未找到五阶段轨迹分类表：", trajectory_file, call. = FALSE)
}

read_deseq_file <- function(one_file, source_type) {
  one_df <- read.csv(one_file, stringsAsFactors = FALSE, check.names = FALSE)
  required_columns <- c(
    "celltype", "stage", "gene", "baseMean", "log2FoldChange", "lfcSE",
    "stat", "pvalue", "padj"
  )
  missing_columns <- setdiff(required_columns, colnames(one_df))
  if (length(missing_columns) > 0) {
    stop(
      "文件缺少 DESeq2 必要字段：", basename(one_file), "；",
      paste(missing_columns, collapse = ", "),
      call. = FALSE
    )
  }

  expected_suffix <- if (source_type == "embryonic") {
    "_macula_vs_peripheral.csv"
  } else {
    "_Central_vs_peripheral.csv"
  }
  if (!endsWith(basename(one_file), expected_suffix)) {
    stop("文件 contrast 与预期不一致：", basename(one_file), call. = FALSE)
  }
  if (n_distinct(one_df$celltype) != 1 || n_distinct(one_df$stage) != 1) {
    stop("单个 DESeq2 文件必须只包含一个 celltype 和 stage：", basename(one_file), call. = FALSE)
  }

  one_df %>%
    mutate(
      source_type = source_type,
      source_file = basename(one_file),
      expected_contrast = if_else(
        source_type == "embryonic",
        "macula_vs_peripheral",
        "Central_vs_peripheral"
      )
    )
}

deseq_raw <- bind_rows(
  lapply(dev_files, read_deseq_file, source_type = "embryonic"),
  lapply(adult_files, read_deseq_file, source_type = "adult_old")
) %>%
  filter(celltype %in% celltype_order) %>%
  mutate(
    celltype = as.character(celltype),
    stage = factor(as.character(stage), levels = stage_order),
    gene = as.character(gene)
  ) %>%
  arrange(celltype, stage, gene)

expected_stage_table <- bind_rows(lapply(names(available_stages), function(one_celltype) {
  data.frame(
    celltype = one_celltype,
    stage = available_stages[[one_celltype]],
    stringsAsFactors = FALSE
  )
}))

observed_stage_table <- deseq_raw %>%
  distinct(celltype, stage) %>%
  mutate(stage = as.character(stage))

if (!setequal(
  paste(expected_stage_table$celltype, expected_stage_table$stage),
  paste(observed_stage_table$celltype, observed_stage_table$stage)
)) {
  stop("实际 DESeq2 celltype × stage 与预设生命周期可用时期不一致。", call. = FALSE)
}

if (anyDuplicated(deseq_raw[c("celltype", "stage", "gene")]) > 0) {
  stop("原始 DESeq2 文件中存在重复 celltype × stage × gene。", call. = FALSE)
}

# 输出：输入文件、contrast 和每个 celltype × stage 的原始基因数量 QC。
input_qc <- deseq_raw %>%
  group_by(celltype, stage, source_type, source_file, expected_contrast) %>%
  summarise(
    n_raw_rows = n(),
    n_finite_stat_raw = sum(is.finite(stat)),
    .groups = "drop"
  )

In [ ]:
# ---- 3. 统一为 SYMBOL rank ID，并处理重复 ID --------------------------------
deseq_with_id_type <- deseq_raw %>%
  mutate(
    raw_gene = str_trim(gene),
    is_ensembl = str_detect(raw_gene, "^ENSG[0-9]+(?:\\.[0-9]+)?$"),
    ensembl_id = if_else(is_ensembl, str_remove(raw_gene, "\\..*$"), NA_character_)
  )

ensembl_ids <- deseq_with_id_type %>%
  filter(is_ensembl) %>%
  pull(ensembl_id) %>%
  unique()

ensembl_symbol_map <- if (length(ensembl_ids) > 0) {
  AnnotationDbi::select(
    org.Hs.eg.db,
    keys = ensembl_ids,
    keytype = "ENSEMBL",
    columns = "SYMBOL"
  ) %>%
    dplyr::filter(!is.na(SYMBOL), SYMBOL != "") %>%
    dplyr::arrange(ENSEMBL, SYMBOL) %>%
    dplyr::group_by(ENSEMBL) %>%
    dplyr::slice_head(n = 1) %>%
    dplyr::ungroup() %>%
    dplyr::rename(ensembl_id = ENSEMBL, mapped_symbol = SYMBOL)
} else {
  data.frame(ensembl_id = character(), mapped_symbol = character())
}

deseq_symbol <- deseq_with_id_type %>%
  left_join(ensembl_symbol_map, by = "ensembl_id") %>%
  mutate(
    SYMBOL = if_else(is_ensembl, mapped_symbol, raw_gene),
    id_mapping_status = case_when(
      is_ensembl & !is.na(mapped_symbol) ~ "ensembl_mapped_to_SYMBOL",
      is_ensembl & is.na(mapped_symbol) ~ "ensembl_unmapped_excluded",
      !is_ensembl & !is.na(SYMBOL) & SYMBOL != "" ~ "input_SYMBOL_retained",
      TRUE ~ "missing_identifier_excluded"
    )
  )

# 输出：原始 gene 到统一 SYMBOL 的映射及无法映射的 Ensembl ID。

deseq_symbol_valid <- deseq_symbol %>%
  filter(
    id_mapping_status != "ensembl_unmapped_excluded",
    id_mapping_status != "missing_identifier_excluded",
    is.finite(stat),
    !is.na(SYMBOL),
    SYMBOL != ""
  )

duplicate_symbol_qc <- deseq_symbol_valid %>%
  group_by(celltype, stage, SYMBOL) %>%
  mutate(
    n_records_for_SYMBOL = n(),
    lfcSE_for_tie = if_else(is.finite(lfcSE), lfcSE, Inf)
  ) %>%
  arrange(celltype, stage, SYMBOL, desc(abs(stat)), lfcSE_for_tie, raw_gene) %>%
  mutate(duplicate_selection_rank = row_number()) %>%
  ungroup()

# 输出：重复 SYMBOL 的保留决策，duplicate_selection_rank = 1 为进入 rank vector 的记录。

deseq_symbol_unique <- duplicate_symbol_qc %>%
  filter(duplicate_selection_rank == 1) %>%
  dplyr::select(
    celltype, stage, SYMBOL, raw_gene, baseMean, log2FoldChange, lfcSE,
    stat, pvalue, padj, source_type, source_file, expected_contrast
  )

In [ ]:
# ---- 4. 建立每个 celltype 的共同有效 SYMBOL 集 -----------------------------
common_symbol_sets <- lapply(names(available_stages), function(one_celltype) {
  one_stages <- available_stages[[one_celltype]]
  stage_symbols <- lapply(one_stages, function(one_stage) {
    deseq_symbol_unique %>%
      filter(celltype == one_celltype, as.character(stage) == one_stage) %>%
      pull(SYMBOL)
  })
  names(stage_symbols) <- one_stages
  Reduce(intersect, stage_symbols)
})
names(common_symbol_sets) <- names(available_stages)

if (any(lengths(common_symbol_sets) == 0)) {
  stop(
    "以下 celltype 没有跨可用时期共同的有限-stat SYMBOL：",
    paste(names(common_symbol_sets)[lengths(common_symbol_sets) == 0], collapse = ", "),
    call. = FALSE
  )
}

In [ ]:
common_gene_qc <- bind_rows(lapply(names(available_stages), function(one_celltype) {
  one_common <- common_symbol_sets[[one_celltype]]
  one_stages <- available_stages[[one_celltype]]
  bind_rows(lapply(one_stages, function(one_stage) {
    one_n <- deseq_symbol_unique %>%
      filter(celltype == one_celltype, as.character(stage) == one_stage) %>%
      nrow()
    data.frame(
      celltype = one_celltype,
      stage = one_stage,
      n_valid_unique_symbols = one_n,
      n_common_symbols = length(one_common),
      common_gene_retention = length(one_common) / one_n,
      stringsAsFactors = FALSE
    )
  }))
}))

In [ ]:
common_gene_summary <- common_gene_qc %>%
  group_by(celltype) %>%
  summarise(
    n_common_symbols = dplyr::first(n_common_symbols),
    common_gene_retention_min = min(common_gene_retention),
    common_gene_retention_max = max(common_gene_retention),
    .groups = "drop"
  )

In [ ]:
# 输出：各阶段共同基因保留率及每个 celltype 的保留率范围。
write.csv(
  bind_rows(
    common_gene_qc %>% mutate(record_type = "celltype_stage"),
    common_gene_summary %>% mutate(record_type = "celltype_summary")
  ),
  file.path(data_dir, paste0(date_str, "_GSEA_common_gene_retention_QC.csv")),
  row.names = FALSE
)

In [ ]:
# ---- 5. 读取统一版本的 MSigDB gene sets -----------------------------------
get_msigdb_set <- function(database, collection, subcollection) {
  one_set <- tryCatch(
    msigdbr::msigdbr(
      species = "Homo sapiens",
      category = collection,
      subcategory = subcollection
    ),
    error = function(e) NULL
  )
  if (is.null(one_set) || nrow(one_set) == 0) {
    warning("当前 msigdbr 版本不包含该 collection，跳过：", database, call. = FALSE)
    return(NULL)
  }
  if (!all(c("gs_name", "gene_symbol") %in% colnames(one_set))) {
    stop("MSigDB gene-set 字段不完整：", database, call. = FALSE)
  }
  one_set %>%
    dplyr::transmute(
      database = database,
      pathway = gs_name,
      SYMBOL = gene_symbol,
      msigdb_version = if ("db_version" %in% colnames(one_set)) db_version else NA_character_
    ) %>%
    dplyr::filter(!is.na(SYMBOL), SYMBOL != "") %>%
    dplyr::distinct()
}

gene_set_catalog <- dplyr::bind_rows(Filter(Negate(is.null), list(
  get_msigdb_set("GO_BP", "C5", "GO:BP"),
  get_msigdb_set("Reactome", "C2", "CP:REACTOME"),
  get_msigdb_set("KEGG_Legacy", "C2", "CP:KEGG_LEGACY"),
  get_msigdb_set("KEGG_Medicus", "C2", "CP:KEGG_MEDICUS")
)))

if (nrow(gene_set_catalog) == 0) {
  stop("当前 msigdbr 版本未返回任何计划分析的 gene set。", call. = FALSE)
}

all_msigdb_symbols <- unique(gene_set_catalog$SYMBOL)
# 输出：本次实际使用的 MSigDB gene-set catalog 和软件版本。

In [ ]:
# ---- 6. 对共同 SYMBOL 的 stat 排序，并确定性处理 ties ----------------------
rank_input <- bind_rows(lapply(names(available_stages), function(one_celltype) {
  deseq_symbol_unique %>%
    filter(
      celltype == one_celltype,
      SYMBOL %in% common_symbol_sets[[one_celltype]]
    )
})) %>%
  mutate(in_any_msigdb_pathway = SYMBOL %in% all_msigdb_symbols)

rank_input <- rank_input %>%
  group_by(celltype, stage) %>%
  arrange(desc(stat), SYMBOL, .by_group = TRUE) %>%
  mutate(
    n_unique_stat = n_distinct(stat),
    min_positive_stat_gap = {
      one_values <- sort(unique(stat))
      one_gaps <- diff(one_values)
      one_positive_gaps <- one_gaps[is.finite(one_gaps) & one_gaps > 0]
      if (length(one_positive_gaps) == 0) NA_real_ else min(one_positive_gaps)
    },
    tie_step = case_when(
      is.finite(min_positive_stat_gap) ~ min_positive_stat_gap / (10 * n()),
      TRUE ~ 1e-10 / max(1, n())
    )
  ) %>%
  group_by(celltype, stage, stat) %>%
  mutate(
    tie_n = n(),
    tie_rank = row_number(),
    stat_for_gsea = stat + (tie_n - tie_rank - (tie_n - 1) / 2) * tie_step
  ) %>%
  ungroup() %>%
  arrange(celltype, stage, desc(stat_for_gsea), SYMBOL)

tie_qc <- rank_input %>%
  group_by(celltype, stage) %>%
  summarise(
    n_ranked_symbols = n(),
    n_msigdb_matched_symbols = sum(in_any_msigdb_pathway),
    n_msigdb_unmatched_symbols_retained = sum(!in_any_msigdb_pathway),
    n_tie_groups = n_distinct(stat[tie_n > 1]),
    n_tied_symbols = sum(tie_n > 1),
    ties_adjusted = any(tie_n > 1),
    rank_is_strictly_decreasing = all(diff(stat_for_gsea) < 0),
    .groups = "drop"
  )

if (!all(tie_qc$rank_is_strictly_decreasing)) {
  stop("ties 调整后仍存在非严格递减的 GSEA 排序向量。", call. = FALSE)
}

# 输出：实际进入 GSEA 的共同 SYMBOL 排序输入，未匹配 MSigDB 的 SYMBOL 仍保留。

# 输出：ties 和 MSigDB 覆盖率 QC。

In [ ]:
# ---- 7. 构建 pathway testability，并验证跨阶段一致性 ----------------------
pathway_sizes <- gene_set_catalog %>%
  count(database, pathway, name = "n_genes_in_catalog")

In [ ]:
pathway_testability <- bind_rows(lapply(names(available_stages), function(one_celltype) {
  one_symbols <- common_symbol_sets[[one_celltype]]
  one_testability <- gene_set_catalog %>%
    group_by(database, pathway) %>%
    summarise(
      n_rank_overlap = n_distinct(SYMBOL[SYMBOL %in% one_symbols]),
      .groups = "drop"
    ) %>%
    left_join(pathway_sizes, by = c("database", "pathway")) %>%
    mutate(
      celltype = one_celltype,
      testability_status = case_when(
        n_rank_overlap < min_size ~ "not_testable_below_minSize",
        n_rank_overlap > max_size ~ "not_testable_above_maxSize",
        TRUE ~ "testable"
      )
    )

    bind_rows(lapply(stage_order, function(one_stage) {
      one_testability %>%
        mutate(
          stage = one_stage,
          stage_status = if (
            one_stage %in% available_stages[[one_celltype]]
          ) {
            testability_status
          } else {
            "not_applicable"
          }
        )
    }))
})) %>%
  dplyr::select(celltype, stage, database, pathway, n_genes_in_catalog,
         n_rank_overlap, testability_status, stage_status)

In [ ]:
testability_check <- pathway_testability %>%
  filter(stage_status != "not_applicable") %>%
  group_by(celltype, database, pathway) %>%
  summarise(
    n_distinct_overlap = n_distinct(n_rank_overlap),
    n_distinct_testability = n_distinct(testability_status),
    testability_consistent_across_available_stages =
      n_distinct_overlap == 1 && n_distinct_testability == 1,
    .groups = "drop"
  )

if (!all(testability_check$testability_consistent_across_available_stages)) {
  stop("共同 rank vector 下 pathway testability 跨可用阶段不一致。", call. = FALSE)
}

# 输出：每个 pathway 的 testability 及跨可用阶段一致性验证。

In [ ]:
# ---- 8. 分 celltype × stage × database 运行 preranked GSEA ----------------
pathway_lists <- split(
  gene_set_catalog$SYMBOL,
  paste(gene_set_catalog$database, gene_set_catalog$pathway, sep = "|||")
)

gsea_results_list <- list()
gsea_errors <- list()

for (one_celltype in names(available_stages)) {
  for (one_stage in available_stages[[one_celltype]]) {
    one_rank_df <- rank_input %>%
      filter(celltype == one_celltype, as.character(stage) == one_stage) %>%
      arrange(desc(stat_for_gsea), SYMBOL)
    one_stats <- setNames(one_rank_df$stat_for_gsea, one_rank_df$SYMBOL)

    for (one_database in unique(gene_set_catalog$database)) {
      one_pathways <- pathway_lists[
        startsWith(names(pathway_lists), paste0(one_database, "|||"))
      ]
      names(one_pathways) <- str_remove(names(one_pathways), paste0("^", one_database, "\\|\\|\\|"))
      one_testable <- pathway_testability %>%
        filter(
          celltype == one_celltype,
          stage == one_stage,
          database == one_database,
          stage_status == "testable"
        ) %>%
        pull(pathway)
      one_pathways <- one_pathways[names(one_pathways) %in% one_testable]

      if (length(one_pathways) == 0) {
        gsea_errors[[length(gsea_errors) + 1]] <- data.frame(
          celltype = one_celltype,
          stage = one_stage,
          database = one_database,
          gsea_run_status = "not_run_no_testable_pathway",
          error_message = NA_character_,
          stringsAsFactors = FALSE
        )
        next
      }

      one_result <- tryCatch(
        fgsea::fgseaMultilevel(
          pathways = one_pathways,
          stats = one_stats,
          minSize = min_size,
          maxSize = max_size
        ),
        error = function(e) e
      )

      if (inherits(one_result, "error")) {
        gsea_errors[[length(gsea_errors) + 1]] <- data.frame(
          celltype = one_celltype,
          stage = one_stage,
          database = one_database,
          gsea_run_status = "gsea_error",
          error_message = conditionMessage(one_result),
          stringsAsFactors = FALSE
        )
        next
      }

      one_result <- as.data.frame(one_result)
      if (!"log2err" %in% colnames(one_result)) one_result$log2err <- NA_real_
      one_result <- one_result %>%
        transmute(
          celltype = one_celltype,
          stage = one_stage,
          database = one_database,
          pathway,
          pval,
          padj,
          log2err,
          ES,
          NES,
          size,
          leadingEdge = vapply(leadingEdge, paste, collapse = ";", character(1)),
          gsea_run_status = "completed"
        )
      gsea_results_list[[length(gsea_results_list) + 1]] <- one_result
    }
  }
}

In [ ]:
gsea_completed <- if (length(gsea_results_list) > 0) {
  bind_rows(gsea_results_list)
} else {
  data.frame()
}
gsea_run_qc <- if (length(gsea_errors) > 0) {
  bind_rows(gsea_errors)
} else {
  data.frame(
    celltype = character(),
    stage = character(),
    database = character(),
    gsea_run_status = character(),
    error_message = character()
  )
}

if (nrow(gsea_completed) == 0) {
  stop("没有任何 celltype × stage × database 完成 GSEA。", call. = FALSE)
}

gsea_status <- pathway_testability %>%
  left_join(
    gsea_completed,
    by = c("celltype", "stage", "database", "pathway")
  ) %>%
  left_join(
    gsea_run_qc %>%
      filter(gsea_run_status == "gsea_error") %>%
      transmute(
        celltype,
        stage,
        database,
        gsea_error_message = error_message
      ),
    by = c("celltype", "stage", "database")
  ) %>%
  mutate(
    gsea_status = case_when(
      stage_status == "not_applicable" ~ "not_applicable",
      stage_status != "testable" ~ "not_testable",
      !is.na(gsea_error_message) ~ "gsea_error",
      gsea_run_status == "completed" & is.finite(padj) & padj < padj_cutoff & NES > 0 ~
        "significant_Central_macular",
      gsea_run_status == "completed" & is.finite(padj) & padj < padj_cutoff & NES < 0 ~
        "significant_Peripheral",
      gsea_run_status == "completed" ~ "tested_not_significant",
      TRUE ~ "gsea_error"
    ),
    direction = case_when(
      NES > 0 ~ "Central_macular",
      NES < 0 ~ "Peripheral",
      TRUE ~ NA_character_
    )
  )

# 输出：全部 GSEA 结果及 tested-not-significant/not-testable 状态。
write.csv(
  gsea_status,
  file.path(data_dir, paste0(date_str, "_lifecycle_directional_GSEA_all_results.csv")),
  row.names = FALSE
)

In [ ]:
# ---- 10. 按 strict/supportive 规则分类 pathway trajectory -----------------
classify_pathway <- function(one_df, one_celltype, is_supportive = FALSE) {
  one_df <- one_df %>%
    filter(stage %in% available_stages[[one_celltype]]) %>%
    mutate(stage = factor(stage, levels = stage_order)) %>%
    arrange(stage)
  one_stages <- as.character(one_df$stage)
  embryo_stages <- intersect(c("Early", "Mid", "Late"), one_stages)
  young_row <- one_df %>% filter(stage == "Young")
  old_row <- one_df %>% filter(stage == "Old")

  if (nrow(young_row) != 1 || nrow(old_row) != 1) {
    return(data.frame(
      strict_trajectory_class = "Undetermined",
      strict_trajectory_direction = NA_character_,
      supportive_trajectory_class = "Undetermined",
      supportive_trajectory_direction = NA_character_,
      abs_NES_Young = NA_real_,
      abs_NES_Old = NA_real_,
      stringsAsFactors = FALSE
    ))
  }

  one_direction_results <- lapply(c("Central_macular", "Peripheral"), function(one_direction) {
    one_sig <- if (one_direction == "Central_macular") {
      "significant_Central_macular"
    } else {
      "significant_Peripheral"
    }
    other_sig <- if (one_direction == "Central_macular") {
      "significant_Peripheral"
    } else {
      "significant_Central_macular"
    }
    one_status <- one_df$gsea_status
    one_stage_names <- as.character(one_df$stage)
    has_opposite <- any(one_status == other_sig)
    has_not_testable <- any(one_status %in% c("not_testable", "gsea_error"))
    embryo_index <- which(one_stage_names %in% embryo_stages)
    young_index <- which(one_stage_names == "Young")
    old_index <- which(one_stage_names == "Old")
    embryo_sig_index <- embryo_index[one_status[embryo_index] == one_sig]
    young_is_sig <- one_status[young_index] == one_sig
    old_is_sig <- one_status[old_index] == one_sig
    old_is_nonsig <- one_status[old_index] == "tested_not_significant"
    nes_drop <- abs(young_row$NES) - abs(old_row$NES)
    is_attenuated <- is.finite(nes_drop) && nes_drop >= min_abs_nes_drop

    strict_class <- NA_character_
    if (!has_not_testable && !has_opposite) {
      if (length(embryo_sig_index) > 0 && young_is_sig && old_is_sig) {
        onset_index <- min(embryo_sig_index)
        if (all(one_status[onset_index:old_index] == one_sig)) {
          strict_class <- if (is_attenuated) {
            "Developmental attenuated-with-age-like"
          } else {
            "Developmental maintained-like"
          }
        }
      }
      if (is.na(strict_class) && length(embryo_sig_index) > 0 && young_is_sig && old_is_nonsig && is_attenuated) {
        onset_index <- min(embryo_sig_index)
        if (all(one_status[onset_index:young_index] == one_sig)) {
          strict_class <- "Developmental aging-lost-like"
        }
      }
      if (
        is.na(strict_class) && young_is_sig && old_is_sig &&
          all(one_status[embryo_index] == "tested_not_significant")
      ) {
        strict_class <- "Adult acquired and maintained-like"
      }
    }

    supportive_class <- NA_character_
    if (!has_opposite) {
      if (length(embryo_sig_index) > 0 && young_is_sig && old_is_sig) {
        supportive_class <- if (is_attenuated) {
          "Developmental attenuated-with-age-like"
        } else {
          "Developmental maintained-like"
        }
      }
      if (is.na(supportive_class) && length(embryo_sig_index) > 0 && young_is_sig && old_is_nonsig && is_attenuated) {
        supportive_class <- "Developmental aging-lost-like"
      }
      embryo_has_any_test <- any(one_status[embryo_index] == "tested_not_significant")
      embryo_has_any_sig <- any(one_status[embryo_index] %in% c(one_sig, other_sig))
      if (is.na(supportive_class) && young_is_sig && old_is_sig && embryo_has_any_test && !embryo_has_any_sig) {
        supportive_class <- "Adult acquired and maintained-like"
      }
    }

    data.frame(
      trajectory_direction = one_direction,
      strict_candidate = strict_class,
      supportive_candidate = supportive_class,
      has_not_testable = has_not_testable,
      has_opposite_significant_NES = has_opposite,
      abs_NES_Young = abs(young_row$NES),
      abs_NES_Old = abs(old_row$NES),
      abs_NES_Young_minus_Old = nes_drop,
      stringsAsFactors = FALSE
    )
  }) %>% bind_rows()

  strict_hit <- one_direction_results %>% filter(!is.na(strict_candidate))
  supportive_hit <- one_direction_results %>% filter(!is.na(supportive_candidate))
  has_any_not_testable <- any(one_direction_results$has_not_testable)

  strict_class <- if (nrow(strict_hit) == 1) strict_hit$strict_candidate else if (has_any_not_testable) "Undetermined" else "Other"
  strict_direction <- if (nrow(strict_hit) == 1) strict_hit$trajectory_direction else NA_character_
  supportive_class <- if (nrow(supportive_hit) == 1) supportive_hit$supportive_candidate else if (has_any_not_testable) "Undetermined" else "Other"
  supportive_direction <- if (nrow(supportive_hit) == 1) supportive_hit$trajectory_direction else NA_character_

    data.frame(
      strict_trajectory_class = strict_class,
      strict_trajectory_direction = strict_direction,
      supportive_trajectory_class = supportive_class,
      supportive_trajectory_direction = supportive_direction,

      abs_NES_Young = if (
        is.finite(young_row$NES)
      ) {
        abs(young_row$NES)
      } else {
        NA_real_
      },

      abs_NES_Old = if (
        is.finite(old_row$NES)
      ) {
        abs(old_row$NES)
      } else {
        NA_real_
      },

      stringsAsFactors = FALSE
    )
}

trajectory_classification <- gsea_status %>%
  filter(stage %in% unlist(available_stages, use.names = FALSE)) %>%
  group_by(celltype, database, pathway) %>%
  group_modify(~ classify_pathway(.x, .y$celltype[[1]], is_supportive = FALSE)) %>%
  ungroup() %>%
  mutate(
    strict_matched_gene_subclass = case_when(
      strict_trajectory_class %in% c(
        "Developmental maintained-like",
        "Developmental attenuated-with-age-like"
      ) ~ "Developmental maintained",
      strict_trajectory_class == "Developmental aging-lost-like" ~ "Developmental aging-lost",
      strict_trajectory_class == "Adult acquired and maintained-like" ~ "Adult acquired and maintained",
      TRUE ~ NA_character_
    ),
    supportive_matched_gene_subclass = case_when(
      supportive_trajectory_class %in% c(
        "Developmental maintained-like",
        "Developmental attenuated-with-age-like"
      ) ~ "Developmental maintained",
      supportive_trajectory_class == "Developmental aging-lost-like" ~ "Developmental aging-lost",
      supportive_trajectory_class == "Adult acquired and maintained-like" ~ "Adult acquired and maintained",
      TRUE ~ NA_character_
    )
  )

# 输出：每个 pathway 的 strict 与 supportive 生命周期分类。
write.csv(
  trajectory_classification,
  file.path(data_dir, paste0(date_str, "_GSEA_strict_supportive_trajectory_classification.csv")),
  row.names = FALSE
)

In [ ]:
# ---- 11. 整合 leading edge 与同方向 threshold gene set --------------------
trajectory_df <- read.csv(
  trajectory_file,
  stringsAsFactors = FALSE,
  check.names = FALSE
)
required_trajectory_columns <- c("celltype", "gene", "subclass", "direction_class")
missing_trajectory_columns <- setdiff(required_trajectory_columns, colnames(trajectory_df))
if (length(missing_trajectory_columns) > 0) {
  stop(
    "五阶段分类表缺少必要字段：",
    paste(missing_trajectory_columns, collapse = ", "),
    call. = FALSE
  )
}

threshold_genes <- trajectory_df %>%
  filter(
    celltype %in% celltype_order,
    subclass %in% target_subclasses,
    direction_class %in% unname(direction_labels),
    !is.na(gene),
    gene != ""
  ) %>%
  mutate(
    raw_gene = str_trim(as.character(gene)),
    is_ensembl = str_detect(raw_gene, "^ENSG[0-9]+(?:\\.[0-9]+)?$"),
    ensembl_id = if_else(is_ensembl, str_remove(raw_gene, "\\..*$"), NA_character_)
  ) %>%
  left_join(ensembl_symbol_map, by = "ensembl_id") %>%
  mutate(SYMBOL = if_else(is_ensembl, mapped_symbol, raw_gene)) %>%
  filter(!is.na(SYMBOL), SYMBOL != "") %>%
  distinct(celltype, subclass, direction_class, SYMBOL)

classification_long <- bind_rows(
  trajectory_classification %>%
    transmute(
      classification_set = "strict",
      celltype, database, pathway,
      trajectory_class = strict_trajectory_class,
      trajectory_direction = strict_trajectory_direction,
      matched_gene_subclass = strict_matched_gene_subclass
    ),
  trajectory_classification %>%
    transmute(
      classification_set = "supportive",
      celltype, database, pathway,
      trajectory_class = supportive_trajectory_class,
      trajectory_direction = supportive_trajectory_direction,
      matched_gene_subclass = supportive_matched_gene_subclass
    )
) %>%
  filter(!is.na(matched_gene_subclass), !is.na(trajectory_direction))

leading_edge_summary_list <- list()
leading_edge_gene_list <- list()

for (i in seq_len(nrow(classification_long))) {
  one_class <- classification_long[i, ]
  one_sig_status <- if (one_class$trajectory_direction == "Central_macular") {
    "significant_Central_macular"
  } else {
    "significant_Peripheral"
  }
  one_direction_class <- direction_labels[[one_class$trajectory_direction]]
  one_result_rows <- gsea_status %>%
    filter(
      celltype == one_class$celltype,
      database == one_class$database,
      pathway == one_class$pathway,
      gsea_status == one_sig_status
    ) %>%
    dplyr::select(stage, NES, padj, leadingEdge)

  one_gene_stage <- bind_rows(lapply(seq_len(nrow(one_result_rows)), function(j) {
    one_genes <- unlist(strsplit(one_result_rows$leadingEdge[j], ";", fixed = TRUE))
    one_genes <- one_genes[one_genes != ""]
    data.frame(
      SYMBOL = one_genes,
      stage = one_result_rows$stage[j],
      stringsAsFactors = FALSE
    )
  }))

  one_threshold <- threshold_genes %>%
    filter(
      celltype == one_class$celltype,
      subclass == one_class$matched_gene_subclass,
      direction_class == one_direction_class
    ) %>%
    pull(SYMBOL) %>%
    unique()

  one_stage_effects <- gsea_status %>%
    filter(
      celltype == one_class$celltype,
      database == one_class$database,
      pathway == one_class$pathway,
      stage %in% available_stages[[one_class$celltype]]
    ) %>%
    dplyr::select(stage, NES, padj, gsea_status) %>%
    mutate(stage = as.character(stage))
  one_effect_values <- c(
    setNames(as.list(one_stage_effects$NES), paste0("NES_", one_stage_effects$stage)),
    setNames(as.list(one_stage_effects$padj), paste0("padj_", one_stage_effects$stage)),
    setNames(as.list(one_stage_effects$gsea_status), paste0("status_", one_stage_effects$stage))
  )

  one_gene_summary <- if (nrow(one_gene_stage) > 0) {
    one_gene_stage %>%
      group_by(SYMBOL) %>%
      summarise(
        leading_edge_stage_count = n_distinct(stage),
        leading_edge_stages = paste(sort(unique(stage), method = "radix"), collapse = ";"),
        in_matching_threshold_gene_set = any(SYMBOL %in% one_threshold),
        .groups = "drop"
      )
  } else {
    data.frame(
      SYMBOL = character(),
      leading_edge_stage_count = integer(),
      leading_edge_stages = character(),
      in_matching_threshold_gene_set = logical()
    )
  }

  one_union <- sort(unique(one_gene_summary$SYMBOL))
  one_overlap <- sort(intersect(one_union, one_threshold))
  one_key <- paste(
    one_class$classification_set, one_class$celltype, one_class$database,
    one_class$pathway, sep = "|||"
  )

  leading_edge_summary_list[[one_key]] <- cbind(
    data.frame(
    classification_set = one_class$classification_set,
    celltype = one_class$celltype,
    database = one_class$database,
    pathway = one_class$pathway,
    trajectory_class = one_class$trajectory_class,
    trajectory_direction = one_class$trajectory_direction,
    matched_gene_subclass = one_class$matched_gene_subclass,
    leading_edge_union = paste(one_union, collapse = ";"),
    n_leading_edge_union = length(one_union),
    leading_edge_total_occurrences = nrow(one_gene_stage),
    matching_threshold_gene_set = paste(sort(one_threshold), collapse = ";"),
    n_matching_threshold_genes = length(one_threshold),
    overlap_with_threshold_genes = paste(one_overlap, collapse = ";"),
    n_overlap_with_threshold_genes = length(one_overlap),
    stringsAsFactors = FALSE
    ),
    as.data.frame(one_effect_values, stringsAsFactors = FALSE)
  )

  if (nrow(one_gene_summary) > 0) {
    leading_edge_gene_list[[one_key]] <- one_gene_summary %>%
      mutate(
        classification_set = one_class$classification_set,
        celltype = one_class$celltype,
        database = one_class$database,
        pathway = one_class$pathway,
        trajectory_class = one_class$trajectory_class,
        trajectory_direction = one_class$trajectory_direction,
        matched_gene_subclass = one_class$matched_gene_subclass,
        .before = 1
      )
  }
}

leading_edge_summary <- if (length(leading_edge_summary_list) > 0) {
  bind_rows(leading_edge_summary_list)
} else {
  data.frame()
}
leading_edge_gene_summary <- if (length(leading_edge_gene_list) > 0) {
  bind_rows(leading_edge_gene_list)
} else {
  data.frame()
}

# 输出：strict/supportive 通路的 leading-edge 并集、重复出现次数和 threshold gene 交集。
write.csv(
  leading_edge_summary,
  file.path(data_dir, paste0(date_str, "_GSEA_leading_edge_threshold_gene_overlap.csv")),
  row.names = FALSE
)

# 输出：leading-edge 单基因跨阶段出现次数、出现阶段和 threshold gene 标记。
write.csv(
  leading_edge_gene_summary,
  file.path(data_dir, paste0(date_str, "_GSEA_leading_edge_gene_stage_summary.csv")),
  row.names = FALSE
)

In [ ]:
# ---- 12. 最终验证与完成信息 ------------------------------------------------
if (anyDuplicated(rank_input[c("celltype", "stage", "SYMBOL")]) > 0) {
  stop("最终 GSEA rank input 中存在重复 celltype × stage × SYMBOL。", call. = FALSE)
}
if (any(gsea_status$gsea_status == "significant_Central_macular" & gsea_status$NES <= 0, na.rm = TRUE) ||
    any(gsea_status$gsea_status == "significant_Peripheral" & gsea_status$NES >= 0, na.rm = TRUE)) {
  stop("GSEA 显著性方向标签与 NES 符号不一致。", call. = FALSE)
}

cat("生命周期分方向 GSEA 完成。\n")
cat("共同 SYMBOL 数：\n")
print(common_gene_summary)
cat("GSEA 结果行数：", nrow(gsea_status), "\n")
cat("输出目录：", data_dir, "\n")